# Yandex AI Studio: Responses API и OpenAI Agents SDK

Это руководство демонстрирует работу с моделями в AI Studio с использованием API, совместимых с OpenAI. Сначала мы выполним прямые запросы к языковым и визуально-языковым моделям, затем добавим встроенные инструменты Web Search и Code Interpreter, а в конце создадим потокового агента для глубокого исследования, который подготовит презентацию и отчёт.

К концу работы с ноутбуком вы научитесь:

1. Подключать клиент OpenAI для Python к Yandex AI Studio.
2. Обращаться к текстовым и визуально-языковым моделям через Responses API.
3. Генерировать программный и HTML-код на основе изображения сайта с помощью VLM.
4. Дополнять ответ данными Web Search.
5. Генерировать и выполнять Python-код с помощью Code Interpreter.
6. Создавать и отслеживать процесс глубокого исследования с помощью OpenAI Agents SDK.
7. Скачивать созданные в ходе работы артефакты в форматах CSV, PNG, PPTX, DOCX и JSON.

LLM и агентные фреймворки OpenAI являются отраслевым стандартом *де-факто* для программной работы с LLM. 
К счастью, Yandex AI Studio создавалась с учётом полной совместимости с OpenAI, поэтому мы можем использовать все инструменты и платформы, разработанные для работы с OpenAI.

В собственных инструментах OpenAI есть два уровня абстракции:

* [Библиотека OpenAI API для Python](https://github.com/openai/openai-python) предоставляет Python-интерфейс к основным API, совместимым с OpenAI. Самый новый из этих API, который мы рассмотрим в этом ноутбуке — [Responses API](https://developers.openai.com/api/reference/responses/overview). Он предоставляет множество продвинутых агентных возможностей: вызов инструментов, структурированный вывод, встроенные инструменты веб-поиска, поиска по файлам и интерпретации кода, имеет поддержку MCP и многое другое.
* [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) добавляет поверх Responses API ещё один, явный уровень агентного цикла и предоставляет абстракции для создания долгоживущих агентов и мультиагентных систем.

В этом ноутбуке мы создадим простого агента анализа данных, способного исследовать произвольные файлы и наборы данных в форматах CSV/XLSX.

## 1. Установка локальных зависимостей

Локальному ядру ноутбука нужны клиенты OpenAI, Agents SDK, поддержка `.env` и библиотека Pillow для отображения изображений. Анализ данных и создание документов Office выполняются во встроенном Code Interpreter в облаке Yandex Cloud, поэтому библиотеки `pandas`, `matplotlib`, `python-pptx` и `python-docx` намеренно не устанавливаются локально.
В образе, созданном для воркшопа SCALE, все необходимые библиотеки установлены.

## 2. Подключение к Yandex AI Studio

AI Studio предоставляет совместимый с OpenAI endpoint по адресу `https://ai.api.cloud.yandex.net/v1`. Для аутентификации нужны два значения, которые мы будем хранить в файле `.env` рядом с этим ноутбуком:

```text
folder_id=...
api_key=...
```

Вспомогательная функция ниже находит корень репозитория независимо от того, запущен ли Jupyter из корневого каталога или из `notebooks/`. Она также проверяет наличие необходимых файлов.

Для работы с низкоуровневым OpenAI Responses API мы используем объект `client`. Мы создаём синхронный и асинхронный клиенты: в примерах прямого обращения к Responses API используется синхронный клиент, а позже в Agents SDK будет использоваться асинхронный.

Для начала импортируем все библиотеки, которые нам пригодятся:

In [ ]:
import base64
import html
import io
import json
import os
import requests
from datetime import date, datetime
from pathlib import Path
import zipfile

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from openai import AsyncOpenAI, OpenAI
from PIL import Image as PILImage
from pydantic import BaseModel, Field
from typing import List
import asyncio
from concurrent.futures import ThreadPoolExecutor
from playwright.async_api import async_playwright
from pydantic import BaseModel, Field
from yhelpers.responses.streaming import jstream

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / '.env').is_file() and (candidate / 'images' / 'old-yandex.png').is_file():
            return candidate
    print('WARNING: Could not find a project root, using current dir')
    return current

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / '.env', override=True)
folder_id = os.environ.get('folder_id')
api_key = os.environ.get('api_key')
if not folder_id or not api_key:
    raise RuntimeError('The .env file must define both folder_id and api_key.')

AI_STUDIO_BASE_URL = 'https://ai.api.cloud.yandex.net/v1'
TEXT_MODEL = f'gpt://{folder_id}/deepseek-v4-flash'
FAST_MODEL = f'gpt://{folder_id}/qwen3.6-35b-a3b/latest'
VISION_MODEL = f'gpt://{folder_id}/qwen3.6-35b-a3b/latest'
IMAGE_MODEL = f'art://{folder_id}/aliceai-image-art-3.0/latest'

client = OpenAI(
    base_url=AI_STUDIO_BASE_URL,
    api_key=api_key,
    project=folder_id,
)
async_client = AsyncOpenAI(
    base_url=AI_STUDIO_BASE_URL,
    api_key=api_key,
    project=folder_id,
)

def print_markdown(text: str) -> None:
    display(Markdown(text))


print(f'Connected configuration for folder: {folder_id[:8]}...')
print(f'Artifacts will be saved under: {OUTPUT_DIR}')

## 3. Подключение к LLM через Responses API

Для базового вызова Responses API нужны модель и входные данные. В качестве примера рассмотрим задачу, в которой требуется сравнить номинальный ВВП России и Китая в текущих долларах США за период с 2016 по 2026 год.

Мы можем попросить агента составить такую таблицу, но без каких-либо инструментов результат, скорее всего, будет галлюцинацией (пусть и довольно убедительно выглядящей). Воспринимайте его как демонстрацию форматирования и знаний модели, а не как достоверный набор экономических данных.

Мы также для скорости выключим рассуждения:

In [ ]:
gdp_question = """
Сравни ВВП России и Китая за годы с 2016 по 2026 включительно.
Выражай ВВП в текущих долларах США, в миллиардах, с 
округлением до одной цифры после запятой.
"""

instructions = """
Ты - аналитик данных. Помогай пользователю анализировать данные.
По необходимости используй Markdown-таблицы для преставления данных.
"""

gdp_response = client.responses.create(
    model=FAST_MODEL,
    instructions=instructions,
    input=gdp_question,
    reasoning = { "effort" : "none" }
)

print_markdown(gdp_response.output_text)

Чтобы продолжить диалог, можно передать параметр `previous_response_id` в тот же вызов:

In [ ]:
followup_response = client.responses.create(
    model=FAST_MODEL,
    instructions=instructions,
    input="Добавь колонку для ВВП США",
    previous_response_id=gdp_response.id,
    reasoning = { "effort" : "none" }
)

print_markdown(followup_response.output_text)

Чтобы не ждать ответа долго, можно использовать режим стриминга - в этом случае токены возвращаются постепенно, в виде событий. События типа "дельта" представляют собой очередные порции сгенерированных токенов:

In [ ]:
stream_response = client.responses.create(
    model=FAST_MODEL,
    instructions=instructions,
    input=gdp_question,
    reasoning = { "effort" : "none" },
    stream=True
)

for evt in stream_response:
    if getattr(evt, 'delta', None):
        print(evt.delta,end='', flush=True)
    else:
        print(evt.type)

Для красивого показа вывода агентов с переформатированием Markdown на лету можно использовать библиотеку [yhelpers](https://github.com/yandex-ai-studio/yhelpers) и функцию `jstream`:

In [ ]:
stream_response = client.responses.create(
    model=FAST_MODEL,
    instructions=instructions,
    input=gdp_question,
    reasoning = { "effort" : "low" },
    stream=True
)

jstream(stream_response)

## 4. Мультимодальные LLM, VLM и другие модели

Визуально-языковые модели принимают текст и изображения в одном пользовательском сообщении. Попробуем дать модели сложную задачу - написать HTML-код для веб-сайта по изображению. Возьмём для эксперимента старый сайт Yandex:

In [ ]:
img_url = "https://raw.githubusercontent.com/yandex-ai-studio/building-data-agent/main/images/old-yandex.png"
site_img = PILImage.open(io.BytesIO(requests.get(img_url).content))
site_img



Для передачи изображений в OpenAI API можно использовать доступную из интернет URL, или закодировать байты изображения в  base64 URL. MIME-тип должен соответствовать формату файла.

In [ ]:
def image_to_data_url(image):
    buffer = io.BytesIO()
    image.save(buffer, format='PNG')
    encoded = base64.b64encode(buffer.getvalue()).decode('ascii')
    return f'data:image/png;base64,{encoded}'

image_to_data_url(site_img)[:100]

Теперь передадим это изображение VLM и попросим сгенерировать HTML-код:

In [ ]:
vision_prompt = """
Recreate the reference screenshot as one complete HTML document. Match the layout, spacing, typography, colors, borders, and visible text.
Use only HTML and inline CSS: no JavaScript, external assets, fonts, or URLs.
Return only the complete HTML document beginning with <!DOCTYPE html>.
"""
def make_input(text, *images):
    content = [{"type": "input_text", "text": text}]
    for image in images:
        content.append(
            {
                "type": "input_image",
                "image_url": image_to_data_url(image),
                "detail": "auto",
            }
        )
    return [{"role": "user", "content": content}]

res = client.responses.create(
    model=VISION_MODEL,
    input=make_input(vision_prompt,site_img)
)

res_html = res.output_text
print(res_html[:100])

Извлечём чистый HTML и посмотрим на него:

In [ ]:
def extract_html(text):
    lowered = text.lower()
    start = lowered.find("<!doctype html")
    if start < 0:
        start = lowered.find("<html")
    end = lowered.rfind("</html>")
    if start < 0 or end < 0:
        raise ValueError("The model did not return a complete HTML document.")
    return text[start : end + len("</html>")].strip()

res_html = extract_html(res_html)
HTML(res_html)

Для оценки качества рендеринга мы можем также использовать VLM. Получим изображение веб-сайта с помощью playwright.

In [ ]:
async def render_html_async(html, width=1280, height=720):
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(
            executable_path=os.environ.get("PLAYWRIGHT_CHROMIUM_EXECUTABLE")
        )
        page = await browser.new_page(
            viewport={"width": width, "height": height},
            device_scale_factor=1,
            java_script_enabled=False,
        )
        await page.set_content(html)
        png_bytes = await page.screenshot(scale="css", type="png")
        await browser.close()

    img = PILImage.open(io.BytesIO(png_bytes))
    img.load()
    return img
def render_html(html, width=1280, height=720):
    def run():
        # Jupyter on Windows uses a Selector loop, which cannot start Chromium.
        loop = asyncio.ProactorEventLoop() if os.name == "nt" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(
                render_html_async(html, width, height)
            )
        finally:
            asyncio.set_event_loop(None)
            loop.close()

    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(run).result()

res_img = render_html(res_html)
res_img

## 5. Структурный ответ

Теперь мы можем сравнить, насколько точно удалось передать дизайн, и сформулировать ряд улучшений. Для этого используем технологию **структурного ответа**, когда мы просим модель вернуть результат в виде заранее описанной схемы:

In [ ]:
class SiteCritique(BaseModel):
    score: int = Field(ge=0, le=100)
    recommendations: list[str]

def critique_images(ref, img):
    prompt = """
The first image is the reference screenshot and the second is an HTML render.
Compare their layout, spacing, typography, colors, borders, and visible text.
Give a strict similarity score from 0 to 100 and a list of concrete changes
that would most improve the next HTML version.
""".strip()

    response = client.responses.parse(
        model=VISION_MODEL,
        input=make_input(prompt, ref, img),
        text_format=SiteCritique,
    )
    return response.output_parsed

res = critique_images(site_img, res_img)
res

Теперь на основе рекомендаций мы можем построить улучшенный вариант дизайна:

In [ ]:
def generate_html(ref_img, previous_html=None, feedback=None):
    previous_html = previous_html or "No previous HTML. Create the first version."
    feedback = feedback.model_dump_json(indent=2) if feedback else "No feedback yet."

    prompt = f"""
Recreate the reference screenshot as one complete HTML document.

Previous HTML:
{previous_html}

Critique:
{feedback}

Match the layout, spacing, typography, colors, borders, and visible text.
Use only HTML and inline CSS: no JavaScript, external assets, fonts, or URLs.
Return only the complete HTML document beginning with <!DOCTYPE html>.
""".strip()

    response = client.responses.create(
        model=VISION_MODEL,
        input=make_input(prompt, ref_img),
    )
    return extract_html(response.output_text)

new_html = generate_html(site_img, res_html, res)
HTML(new_html)


## 6. Agentic Loop. Задание 1

Мы научились:
* генерировать код сайта по входному изображению,
* рендерить сайт
* сравнивать полученное изображение с эталонным и формировать рекомендации по улучшению

Теперь мы можем подать рекомендации на вход первому шагу, и тем самым замкнуть **агентский цикл**, который будет циклически улучшать первоначальную генерацию. Попробуйте реализовать такой агентский цикл самостоятельно, и сгеренировать 3-5 последовательных итераций улучшения дизайна. Используйте определённые выше функции.

> Обратите внимание, что функция `generate_html` реализована таким образом, что может работать и без картинки и обратной связи с предыдущих итераций.

In [ ]:
# Реализуйте агентский цикл

pass

## 7. Вызов инструментов

Идея вызова инструментов состоит в том, чтобы дать LLM возможность обратиться за помощью к некоторому внешнему алгоритму. Модели предоставляется список доступных инструментов, после чего она может не отвечать сразу пользователю, а выбрать один из инструментов для вызова. Агентная обвязка (харнесс) должна вызвать инструмент и передать результат обратно в LLM.

Рассмотрим пример: мы хотим помочь LLM выполнять вычисления. Для этого можно предоставить ей инструмент, вычисляющий произвольные выражения:

In [ ]:
import math

def calculate(expression):
    """
    Safely computes arithmetic expression.
    
    Uses a limited set of functions from the math module
    to avoid executing arbitrary code.
    """
    # Словарь разрешённых функций и констант
    allowed = {
        "sqrt": math.sqrt,
        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,
        "log": math.log,
        "log10": math.log10,
        "log2": math.log2,
        "exp": math.exp,
        "abs": abs,
        "round": round,
        "pi": math.pi,
        "e": math.e,
    }
    
    try:
        # Безопасно вычисляем выражение в ограниченном окружении
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"

calculate("1+2*3")

Чтобы описать этот инструмент для LLM, используем JSON Schema:

In [ ]:
calculator_tool = {
    "type": "function",
    "name": "calculate",
    "description": "Compute mathematical expression. Supports operations (+, -, *, /), exponentiation (**), square root (sqrt), trigonometry (sin, cos, tan), logarithms (log, log10) and constants (pi, e).",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "Mathematical expression in Python, e.g., '2**10', 'sqrt(144)', 'sin(pi/4)'"
            }
        },
        "required": ["expression"]
    }
}

In [ ]:

instruction = """
You are a mathematical assistant. You have a calculator for precise calculations.
When a user presents a task requiring computations:
1. Formulate the mathematical expression
2. Call the calculate function to obtain the exact result
3. Explain the answer to the user

Always use the calculator for computations — don't calculate in your head!
"""

# Отправляем запрос
res = client.responses.create(
    model=FAST_MODEL,
    tools=[calculator_tool],
    instructions=instruction,
    input="Какой квадратный корень из 2026?"
)

# Смотрим, что вернула модель
print("The model's response contains:")
for item in res.output:
    print(f"  - type: {item.type}")
    if item.type == "function_call":
        print(f"    function: {item.name}")
        print(f"    arguments: {item.arguments}")
        print(f"    call_id: {item.call_id}")

Теперь нам нужно вручную вызвать функцию, чтобы выполнить вычисления:

In [ ]:
max_iterations = 5
iteration = 0

while iteration < max_iterations:
    # Ищем вызовы функций в выводе
    tool_calls = [item for item in res.output if item.type == "function_call"]
    
    if not tool_calls:
        # Вызовов нет — модель готова дать окончательный ответ
        break
    
    # Обрабатываем вызовы функций
    outputs = []
    for call in tool_calls:
        args = json.loads(call.arguments) if call.arguments else {}
        expression = args.get("expression", "")
        
        # Выполняем вычисления
        result = calculate(expression)
        print(f"  🔢 calculate({expression}) = {result}")
        
        # Формируем ответ для модели
        outputs.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": result
        })
    
    # Отправляем результаты обратно модели
    res = client.responses.create(
        model=FAST_MODEL,
        tools=[calculator_tool],
        previous_response_id=res.id,
        input=outputs
    )
    
    iteration += 1

print(res.output_text)


AI Studio и Responses API поддерживают несколько типов инструментов:
* инструменты, определённые и обрабатываемые вручную в коде - как в примере выше;
* встроенные инструменты: Web Search, File Search (RAG), Code Interpreter и Image Generation;
* удалённые инструменты, доступные по протоколу MCP.


## 8. Веб-поиск

Встроенный инструмент `web_search` позволяет модели получать актуальную информацию перед ответом. Благодаря этому один запрос к Responses API превращается в простое агентное взаимодействие: модель решает, что искать, изучает результаты и формулирует итоговый ответ.

Мы повторим вопрос о ВВП, но теперь потребуем ссылки на источники, единообразные единицы измерения и явную обработку противоречащих друг другу и прогнозных значений. Полученной таблице следует отдавать предпочтение перед предыдущим ответом, основанным только на знаниях модели, хотя её всё равно нужно сверять с указанными источниками.

In [ ]:
web_search_tool = {
    'type': 'web_search',
    'search_context_size': 'medium',
}

web_question = """
Какие события последнего года оказали влияние на ВВП Китая?
"""

web_response = client.responses.create(
    model=FAST_MODEL,
    instructions=f"""
Ты — внимательный помощник по экономическим исследованиям. Перед ответом используй веб-поиск для поиска актуальной информации. Сейчас {datetime.now()}
""",
    tools=[web_search_tool],
    input=web_question,
    stream=True
)

web_response = jstream(web_response)

В рамках объекта `web_response` также были возвращены все интернет-запросы и использованные при формулировании ответа ресурсы: 

In [ ]:
print("Результаты:")

for item in web_response.output:
    if getattr(item, "type") == "web_search_call":
        if item.action.type == "search":
            queries = getattr(item.action, "queries", None) or [getattr(item.action, "query", None)]
            for q in filter(None, queries):
                print(f" + Query: {q}")
        elif item.action.type == "open_page":
            print(f" + Opened page: {item.action.url}")
    elif getattr(item, "type") == "message":
        for content in getattr(item, "content", []):
            for annotation in getattr(content, "annotations", []):
                print(f" + Title: {getattr(annotation, 'title', 'Untitled')}")
                print(f"   URL: {getattr(annotation, 'url', 'Undefined')}")


## 9. Настоящая магия: Code Interpreter

Настоящая магия анализа данных начинается, когда мы разрешаем LLM писать код: Code Interpreter выполняет воспроизводимые преобразования, вычисления и построение графиков в изолированном встроенном контейнере. Явно указанный идентификатор контейнера упрощает проверку и повторное использование рабочей области в этом разделе.

Модели предписано сначала выполнить поиск, привести все значения к миллиардам долларов США, сохранить URL источников и статус данных, а затем создать два CSV-файла и два PNG-графика. Библиотеки анализа используются или устанавливаются внутри Code Interpreter и не являются зависимостями ядра этого ноутбука.

In [ ]:
analysis_instructions = """
Ты — аналитик данных с доступом к интерпретатору кода. Твоя задача - выполнять запрос пользователя и строить релевантные графики и таблицы. Ты можешь использовать Python, pandas и matplotlib для анализа данных.
"""

gdp_analysis_response = client.responses.create(
    model=TEXT_MODEL,
    store=True,
    instructions=analysis_instructions,
    input='Построй динамику ВВП Китая с 2015 по 2025 г. и процентые доли ВВП по отраслям',
    include=['code_interpreter_call.outputs'],
    tools=[
        {'type': 'code_interpreter', 'container': { "type" : "auto" }},
    ],
    stream=True
)

gdp_analysis_response = jstream(gdp_analysis_response)

### Проверка и скачивание артефактов Code Interpreter

Созданные файлы появляются в выводе сообщений как аннотации `container_file_citation`. Вспомогательная функция ниже также выводит код и журналы Code Interpreter, если они доступны. Она очищает удалённые имена файлов, устраняет повторяющиеся идентификаторы файлов, обрабатывает отдельные ошибки скачивания и возвращает локальные пути к успешно скачанным файлам.

In [ ]:
def download_file_citations(annotations, download_dir: Path, seen_file_ids: set[str]):
    downloaded = []
    for annotation in annotations or []:
        if getattr(annotation, 'type', None) != 'container_file_citation':
            continue
        file_id = getattr(annotation, 'file_id', None)
        filename = Path(getattr(annotation, 'filename', '')).name
        if not file_id or not filename or file_id in seen_file_ids:
            continue
        try:
            payload = client.files.content(file_id)
            local_path = download_dir / filename
            local_path.write_bytes(payload.read())
            downloaded.append(local_path)
            seen_file_ids.add(file_id)
            print(f'Downloaded: {local_path}')
        except Exception as exc:
            print(f'Could not download {filename}: {type(exc).__name__}: {exc}')
    return downloaded


def inspect_and_download_response(response, download_dir: Path):
    download_dir.mkdir(parents=True, exist_ok=True)
    downloaded = []
    seen_file_ids: set[str] = set()
    print('Response items:', [getattr(item, 'type', type(item).__name__) for item in response.output])

    for item in response.output:
        if getattr(item, 'type', None) == 'code_interpreter_call':
            print('\n=== Code Interpreter ===')
            print('Container:', getattr(item, 'container_id', None))
            if getattr(item, 'code', None):
                print('\nExecuted code:\n', item.code)
            for output_item in getattr(item, 'outputs', []) or []:
                logs = getattr(output_item, 'logs', None)
                if logs:
                    print('\nLogs:\n', logs)
        elif getattr(item, 'type', None) == 'message':
            for content in getattr(item, 'content', []) or []:
                downloaded.extend(
                    download_file_citations(
                        getattr(content, 'annotations', []), download_dir, seen_file_ids
                    )
                )
    return downloaded


gdp_downloads = inspect_and_download_response(
    gdp_analysis_response, OUTPUT_DIR / 'gdp_analysis'
)

for local_path in gdp_downloads:
    if local_path.suffix.lower() == '.png':
        with PILImage.open(local_path) as chart:
            display(chart.copy())

Мы также можем загрузить в интерпретатор кода файлы, с которыми нам необходимо работать. В этом случае при создании контейнера мы передаём соответствующие `file_id`s.

Рассмотрим пример анализа CSV-таблицы с ВВП:

In [ ]:
analysis_instructions = """
Ты — аналитик данных с доступом к интерпретатору кода. Твоя задача - выполнять запрос пользователя и строить релевантные графики и таблицы. Ты можешь использовать Python, pandas и matplotlib для анализа данных.
"""

uploaded_files = []
with zipfile.ZipFile(PROJECT_ROOT / "data" / "gdp.zip") as zf:
    for info in zf.infolist():
        print(f"Uploading {info.filename}...")
        data = zf.read(info)
        buf = io.BytesIO(data)
        buf.name = info.filename
        resp = client.files.create(file=buf, purpose="assistants")
        uploaded_files.append(resp)

print("Running...")
gdp_analysis_response = client.responses.create(
    model=TEXT_MODEL,
    store=True,
    instructions=analysis_instructions,
    input='Используй файл gdp.csv и построй сравнительную динамику ВВП Китая и России за доступные годы',
    include=['code_interpreter_call.outputs'],
    tools=[
        {'type': 'code_interpreter', 'container': 
            { 
               "type" : "auto", 
               "file_ids": [x.id for x in uploaded_files] 
            }
        },
    ],
    stream=True
)

gdp_analysis_response = jstream(gdp_analysis_response)

Скачаем результаты и посмотрим на код, сгенерированный моделью:

In [ ]:
gdp_downloads = inspect_and_download_response(
    gdp_analysis_response, OUTPUT_DIR / 'gdp_analysis'
)

for local_path in gdp_downloads:
    if local_path.suffix.lower() == '.png':
        with PILImage.open(local_path) as chart:
            display(chart.copy())

## 10. Shell Tool и Навыки (Skills)

Интерпретатор Python позволяет выполнять программы на Python, но иногда бывает полезным выполнять произвольные команды операционной системы. Для этого в Responses API есть ещё один инструмент - Shell Tool. При этом LLM также генерирует код, который выполняется в контейнере.

Совместно с Shell Tool можно использовать ещё один полезный механизм: навыки (скиллы). Это текстовые файлы, описывающие на естественном языке, как нужно решать определённые задачи. При этом в составе скилла могут быть указания выполнять определённые команды операционной системы или Python-скрипты.

Рассмотрим пример - поиск книг в [библиотеке Мошкова](http://lib.ru). Соответствующий скилл находится в [skills/libru-search](skills/libru-search), и описывает основные правила поиска (например, какую URL запрашивать для поиска по подстроке). Это позволяет агенту тратить меньше токенов, т.к. он уже знает, как действовать, без необходимости исследовать структуру сайта.

В нашем скилле будет текстовая инструкция в файле [SKILL.md](skills/libru-search/SKILL.md), а также пара скриптов для поиска по подстроке и скачивания файла в корректной кодировке.

Для использования навыков нужно сначала загрузить zip-файлы со скиллами в облако, и получить соответствующие ссылки. Для удобства мы запакуем все файлы навыка в архив "на лету":

In [ ]:
def list_files(path):
    files = []
    for file_path in path.rglob("*"):
        if file_path.is_file():
            rel = file_path.relative_to((path / "..").resolve())
            files.append((rel.as_posix(),open(file_path,'rb')))
    return files

skill = client.skills.create(files=list_files(Path(PROJECT_ROOT / "skills/libru-search")))

print(f"Навык {skill} создан")

Теперь вызовем Responses API с данным навыком и Shell Tool:

In [ ]:
res = client.responses.create(
    model=TEXT_MODEL,
    instructions="Ты - библиотекарь, имеющий доступ к библиотеке lib.ru",
    input='Построй частотный словарь упоминания персонажей в книге Анна Каренина',
    tools=[
        {
            "type": "shell",
            "environment": {
                "type": "container_auto",
                "skills": [{
                    "type": "skill_reference",
                    "skill_id": skill.id
                }]
            }
        }],
    stream=True
)
res = jstream(res)

## 11. OpenAI Agents SDK с AI Studio

В приведённых выше примерах Responses API модель использует встроенные инструменты в рамках одного API-запроса. Agents SDK добавляет явный агентный цикл: он многократно вызывает модель, выполняет инструменты, передаёт результаты обратно и останавливается после завершения задачи.

`OpenAIResponsesModel` подключает SDK к клиенту Yandex `AsyncOpenAI`. Рекомендется также отключить встроенную трассировку средствами SDK, поскольку она специфична для OpenAI.

In [ ]:
from agents import (
    Agent,
    CodeInterpreterTool,
    Runner,
    WebSearchTool,
    function_tool,
    set_tracing_disabled,
)
from agents.models.openai_responses import OpenAIResponsesModel
from openai.types.responses import ResponseTextDeltaEvent

set_tracing_disabled(True)

yandex_agent_model = OpenAIResponsesModel(
    model=TEXT_MODEL,
    openai_client=async_client,
)

print(f'Agents SDK model configured: {TEXT_MODEL}')

В качестве демонстрации работы OpenAI Agents SDK реализуем агента для Deep Research. В рамках этого процесса агент будет искать информацию на релевантную тему в интернет, запоминать основные моменты в заметках, и затем генерировать отчёт пользователю.

## 11. Добавление собственных инструментов для заметок

Глубокое исследование может включать множество поисковых запросов и вызовов инструментов. Небольшая внешняя память помогает агенту сохранять результаты и не полагаться только на контекст модели. Следующие функции делают хранящийся в памяти список доступным SDK в виде инструментов. Аннотации типов и строки документации становятся частью схемы каждого инструмента.

Эта учебная реализация существует только в течение текущего сеанса. В промышленной системе обычно используют постоянное хранилище и записывают метаданные источников, временные метки и идентификаторы исследовательских сеансов.

In [ ]:
research_notes: list[dict[str, str]] = []


@function_tool
def save_note(topic: str, content: str, source_url: str = '') -> str:
    """Save one evidence-bearing research note with an optional source URL."""
    research_notes.append(
        {'topic': topic, 'content': content, 'source_url': source_url}
    )
    return f"Saved note '{topic}'. Total notes: {len(research_notes)}"


@function_tool
def get_all_notes() -> str:
    """Return every note saved during the current research run."""
    if not research_notes:
        return 'No notes have been saved.'
    return json.dumps(research_notes, ensure_ascii=False, indent=2)


@function_tool
def clear_notes() -> str:
    """Clear all notes before starting a new research topic."""
    count = len(research_notes)
    research_notes.clear()
    return f'Cleared {count} notes.'

В функциях выше мы используем типизацию для всех функций, потому что она помогает построить правильную JSON-схему для LLM Function Calling.

## 12. Создание агента для глубокого исследования

Агент объединяет три вида возможностей:

- **Web Search** собирает актуальные фактические данные.
- **Инструменты заметок** сохраняют важные результаты и их URL.
- **Code Interpreter** выполняет количественный анализ, создаёт графики и формирует итоговые документы PPTX и DOCX во встроенном контейнере.

Измените `research_topic`, чтобы повторно использовать этот рабочий процесс. Тема по умолчанию связана с применением данных о зданиях. Промпт требует явного плана исследования, заметок с подтверждающими данными, взвешенного рассмотрения неопределённости и двух итоговых документов Office.

In [ ]:
research_topic = 'Как ИИ-агенты меняют разработку ПО в последние месяцы'

research_container = client.containers.create(
    name='deep-research-deliverables',
    expires_after={'anchor': 'last_active_at', 'minutes': 20},
)

deep_research_instructions = f"""
Ты — агент для глубоких исследований Deep Research. Сегодняшняя дата — {date.today().isoformat()}. Твоя задача — провести тщательное исследование по заданной пользователем теме. Ты должен собрать достоверные доказательства, проанализировать их и подготовить профессиональные материалы.

Методология:
1. Начни с составления плана исследования из 4–7 шагов, запиши план.
2. Изучи каждый крупный аспект с помощью нескольких поисковых запросов в интернете. Отдавай предпочтение первичным, авторитетным и свежим источникам.
3. После каждой существенной находки вызывай save_note с краткой темой, саммари находки и URL-адресом источника. Не сохраняй неподтверждённые утверждения как факты.
4. Перед синтезом используй get_all_notes, чтобы не упустить важные доказательства.
5. Используй интерпретатор кода для расчётов, структурированных сравнений и построения графиков.
6. Отделяй факты из источников, оценки, интерпретации и рекомендации.

Результаты:
- Создай профессиональную презентацию PPTX с титульным слайдом, резюме для руководства, методологией, выводами, подкреплёнными доказательствами, как минимум двумя полезными графиками, рекомендациями, ограничениями и слайдом с источниками.
- Создай подробный отчёт DOCX, охватывающий то же исследование, с более полным текстом, ссылками на источники, методологией, выводами, ограничениями, заключениями и рекомендациями.
- При необходимости используй или установи python-pptx, python-docx, pandas и matplotlib внутри интерпретатора кода. Не проси локальное ядро ноутбука создавать документы.
- Используй описательные имена файлов, оканчивающиеся на .pptx и .docx, и прикрепи оба файла к итоговому ответу.
- Верни краткое итоговое резюме со ссылками или цитатами и перечисли все созданные файлы.

Пиши все заметки, текст, подписи к графикам и итоговые материалы на русском языке.
"""

deep_research_agent = Agent(
    name='DeepResearchAgent',
    model=yandex_agent_model,
    instructions=deep_research_instructions,
    tools=[
        WebSearchTool(search_context_size='high'),
        save_note,
        get_all_notes,
        clear_notes,
        CodeInterpreterTool(
            tool_config={'type': 'code_interpreter', 'container': research_container.id}
        ),
    ],
)

print(f'Deep Research agent is ready for: {research_topic}')

## 13. Потоковое выполнение исследования

`Runner.run_streamed` запускает тот же агентный цикл, что и `Runner.run`, но предоставляет асинхронный поток событий. Необработанные события ответа содержат фрагменты текста, а события элементов выполнения более высокого уровня описывают поисковые запросы, вызовы функций, работу Code Interpreter, этапы рассуждения и результаты инструментов.

Обработчик событий ниже намеренно сделан самодостаточным. Он не зависит от состояния исходного ноутбука вебинара и экранирует HTML перед отображением диагностического текста. После завершения потока полный результат остаётся доступным.

In [ ]:
def show_event(label: str, value, color: str = '#1f2937') -> None:
    safe_label = html.escape(str(label))
    safe_value = html.escape(str(value))
    display(
        HTML(
            f"<div style='font-family:monospace;color:{color}'>"
            f"<b>{safe_label}</b>{safe_value}</div>"
        )
    )


async def stream_agent_run(streamed_result) -> None:
    async for event in streamed_result.stream_events():
        if event.type == 'agent_updated_stream_event':
            show_event('Agent: ', event.new_agent.name, '#7c3aed')

        elif event.type == 'raw_response_event' and isinstance(
            event.data, ResponseTextDeltaEvent
        ):
            if event.data.delta:
                print(event.data.delta, end='', flush=True)

        elif event.type == 'run_item_stream_event':
            item = event.item
            raw = getattr(item, 'raw_item', None)
            kind = str(getattr(raw, 'type', type(raw).__name__)).lower()

            if item.type == 'tool_call_item':
                if 'web_search' in kind:
                    action = getattr(raw, 'action', None)
                    show_event('Web Search: ', getattr(action, 'query', ''), '#0369a1')
                elif 'function' in kind:
                    call = f"{getattr(raw, 'name', '?')}({getattr(raw, 'arguments', '')})"
                    show_event('Notes tool: ', call, '#0369a1')
                elif 'code_interpreter' in kind:
                    show_event('Code Interpreter: ', 'running...', '#0369a1')
                else:
                    show_event('Tool: ', kind, '#0369a1')

            elif item.type == 'tool_call_output_item':
                preview = str(getattr(item, 'output', ''))[:700]
                show_event('Tool result: ', preview, '#047857')

            elif item.type == 'reasoning_item':
                summary = getattr(raw, 'summary', None)
                if summary:
                    show_event('Reasoning step: ', summary, '#b45309')

In [ ]:
research_notes.clear()

research_task = f"""
Проведи исследование по теме:
{research_topic}
"""

streamed_research = Runner.run_streamed(
    deep_research_agent,
    research_task,
    max_turns=45,
)
await stream_agent_run(streamed_research)

print('\n\nResearch run completed.')
print_markdown(streamed_research.final_output)

## 14. Скачивание результатов исследования и экспорт заметок

Результаты Agents SDK предоставляют новые сообщения и взаимодействия с инструментами через `new_items`. Ссылки на файлы, встроенные в содержимое сообщений, используют тот же Files API, что и пример прямого обращения к Responses API. Вспомогательная функция повторно использует ранее определённый загрузчик с защитными проверками.

Заметки являются локальными данными Python, поэтому они экспортируются напрямую в JSON. Так вместе с созданными агентом файлами PPTX и DOCX получается проверяемая промежуточная запись.

In [ ]:
def download_agent_artifacts(result, download_dir: Path):
    download_dir.mkdir(parents=True, exist_ok=True)
    downloaded = []
    seen_file_ids: set[str] = set()

    for run_item in result.new_items:
        raw_item = getattr(run_item, 'raw_item', None)
        for content in getattr(raw_item, 'content', []) or []:
            downloaded.extend(
                download_file_citations(
                    getattr(content, 'annotations', []), download_dir, seen_file_ids
                )
            )
    if not downloaded:
        print('No downloadable file annotations were found in the final result.')
    return downloaded


research_output_dir = OUTPUT_DIR / 'deep_research'
research_downloads = download_agent_artifacts(streamed_research, research_output_dir)

notes_path = research_output_dir / 'research_notes.json'
notes_path.write_text(
    json.dumps(research_notes, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'Exported {len(research_notes)} notes to: {notes_path}')

print('\nResearch artifacts:')
for artifact_path in [*research_downloads, notes_path]:
    print(f' - {artifact_path}')

## 15. Итоги

Примеры образуют последовательное развитие решения, а не конкурирующие реализации:

- Используйте **прямой вызов Responses API** для конкретного промпта с предсказуемыми входными и выходными данными.
- Добавляйте **Web Search**, когда важны актуальность и ссылки на источники.
- Добавляйте **Code Interpreter**, когда задача требует надёжных вычислений, преобразований, графиков или создания файлов.
- Используйте **Agents SDK**, когда модель должна планировать работу, многократно вызывать инструменты, поддерживать внешнюю рабочую память, передавать ход выполнения в потоковом режиме и собирать несколько итоговых материалов.

Всегда проверяйте сгенерированные данные и документы. Результаты поиска могут расходиться, прогнозы — меняться, а качественно оформленный график или слайд не гарантирует достоверности лежащих в его основе утверждений. Для промышленного применения добавьте постоянное хранилище заметок, явные правила оценки качества источников, обработку повторных попыток и тайм-аутов, контроль затрат, наблюдаемость и хранение артефактов вне временных контейнеров.